---

#### Environment Setup (Google Colab / Local)

In [ ]:
# ──────────────────────────────────────────────
# Install required packages (run once)
# ──────────────────────────────────────────────
# !pip install -q scikit-learn xgboost lightgbm catboost optuna shap mlflow wandb
# !pip install -q pandas numpy matplotlib seaborn joblib

In [ ]:
# ──────────────────────────────────────────────
# Core Imports
# ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_auc_score,
                             mean_squared_error, mean_absolute_error, r2_score, roc_curve)

# Models
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              GradientBoostingClassifier, GradientBoostingRegressor,
                              AdaBoostClassifier, BaggingClassifier, VotingClassifier)
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA

# Datasets
from sklearn.datasets import (load_iris, load_breast_cancer, load_wine,
                               fetch_california_housing, load_digits, make_classification,
                               make_regression, make_blobs)

print("All imports successful!")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

import sklearn
print(f"Scikit-learn: {sklearn.__version__}")

---

<a id="fundamentals"></a>

### Part 1: ML Fundamentals and Preprocessing

#### Types of Machine Learning

| Type | Description | Examples |
| --- | --- | --- |
| **Supervised** | Labeled data; learns input-output mapping | Classification, Regression |
| **Unsupervised** | No labels; finds hidden patterns | Clustering, Dimensionality Reduction |
| **Semi-supervised** | Mix of labeled + unlabeled data | Label propagation, self-training |
| **Reinforcement** | Agent learns through rewards/penalties | Game AI, robotics, recommendation |

#### Bias-Variance Tradeoff

| Concept | Description | Effect |
| --- | --- | --- |
| **Bias** | Error from oversimplified assumptions | Underfitting (high bias) |
| **Variance** | Error from sensitivity to training data | Overfitting (high variance) |
| **Tradeoff** | Decreasing one typically increases other | Sweet spot = optimal complexity |
| **Bagging** | Reduces variance (Random Forest) | Averaging multiple models |
| **Boosting** | Reduces bias (XGBoost, AdaBoost) | Sequential error correction |

#### Overfitting vs Underfitting

| Issue | Symptom | Solutions |
| --- | --- | --- |
| **Overfitting** | High train accuracy, low test accuracy | Regularization, more data, dropout, early stopping, cross-validation |
| **Underfitting** | Low train AND test accuracy | More features, complex model, less regularization, more training |

#### Data Preprocessing

##### Feature Scaling

| Method | Formula | When to Use |
| --- | --- | --- |
| **StandardScaler** | z = (x - mean) / std | Most ML algorithms, SVM, Logistic Regression |
| **MinMaxScaler** | x' = (x - min) / (max - min) | Neural networks, algorithms sensitive to magnitude |
| **RobustScaler** | x' = (x - median) / IQR | Data with outliers |
| **MaxAbsScaler** | x' = x / max(abs(x)) | Sparse data |
| **Normalizer** | x' = x / norm(x) | Text classification, KNN |

##### Handling Missing Values

| Strategy | Method | When to Use |
| --- | --- | --- |
| **Mean/Median** | `SimpleImputer(strategy='mean')` | Numerical features, few missing |
| **Mode** | `SimpleImputer(strategy='most_frequent')` | Categorical features |
| **KNN Imputer** | `KNNImputer(n_neighbors=5)` | When similar samples exist |
| **Iterative** | `IterativeImputer()` | MICE — multiple rounds of imputation |
| **Drop** | `dropna()` | Very few missing values |

##### Encoding Categorical Variables

| Method | When to Use | Example |
| --- | --- | --- |
| **Label Encoding** | Ordinal features (low/med/high) | `LabelEncoder()` |
| **One-Hot Encoding** | Nominal features (color, city) | `OneHotEncoder()` or `pd.get_dummies()` |
| **Ordinal Encoding** | Known ordering | `OrdinalEncoder(categories=...)` |
| **Target Encoding** | High-cardinality categoricals | Mean of target per category |

In [ ]:
# ──────────────────────────────────────────────
# ML Pipeline Flowchart
# ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 2.5))
steps = ['Raw Data', 'Clean /\nPreprocess', 'Feature\nEngineering', 'Train/Test\nSplit',
         'Model\nTraining', 'Evaluation', 'Deployment']
colors = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6', '#e74c3c', '#1abc9c', '#34495e']

for i, (step, color) in enumerate(zip(steps, colors)):
    x = i * 1.8
    rect = mpatches.FancyBboxPatch((x, 0.2), 1.4, 0.6, boxstyle="round,pad=0.1",
                                    facecolor=color, edgecolor='white', alpha=0.85)
    ax.add_patch(rect)
    ax.text(x + 0.7, 0.5, step, ha='center', va='center', fontsize=8,
            fontweight='bold', color='white')
    if i < len(steps) - 1:
        ax.annotate('', xy=(x + 1.6, 0.5), xytext=(x + 1.4, 0.5),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=2))

ax.set_xlim(-0.3, 12.5)
ax.set_ylim(-0.1, 1.1)
ax.axis('off')
ax.set_title('Machine Learning Pipeline', fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

In [ ]:
# ──────────────────────────────────────────────
# Data Preprocessing Examples
# ──────────────────────────────────────────────

# Load dataset
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
df = data.frame
print("Dataset shape:", df.shape)
print("\nFirst 3 rows:")
print(df.head(3))
print("\nBasic stats:")
print(df.describe().round(2))

In [ ]:
# ──────────────────────────────────────────────
# Feature Scaling Comparison
# ──────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

X = df[['MedInc', 'AveRooms', 'Population']].values[:5]
print("Original:\n", X)

print("\nStandardScaler (zero mean, unit variance):")
print(StandardScaler().fit_transform(X).round(3))

print("\nMinMaxScaler (0 to 1):")
print(MinMaxScaler().fit_transform(X).round(3))

print("\nRobustScaler (resistant to outliers):")
print(RobustScaler().fit_transform(X).round(3))

In [ ]:
# ──────────────────────────────────────────────
# Handling Missing Values
# ──────────────────────────────────────────────
from sklearn.impute import SimpleImputer

# Create sample data with missing values
df_missing = pd.DataFrame({
    'age': [25, np.nan, 30, 35, np.nan],
    'salary': [50000, 60000, np.nan, 80000, 70000],
    'city': ['NYC', 'LA', 'NYC', np.nan, 'LA']
})
print("Data with missing values:")
print(df_missing)

# Impute numerical
num_imputer = SimpleImputer(strategy='mean')
df_missing[['age', 'salary']] = num_imputer.fit_transform(df_missing[['age', 'salary']])

# Impute categorical
cat_imputer = SimpleImputer(strategy='most_frequent')
df_missing[['city']] = cat_imputer.fit_transform(df_missing[['city']])

print("\nAfter imputation:")
print(df_missing)

In [ ]:
# ──────────────────────────────────────────────
# Encoding Categorical Variables
# ──────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# Label Encoding (ordinal)
le = LabelEncoder()
sizes = ['small', 'medium', 'large', 'medium', 'small']
encoded = le.fit_transform(sizes)
print(f"Label Encoding: {sizes} -> {encoded}")
print(f"Classes: {le.classes_}")

# One-Hot Encoding (nominal)
colors = np.array(['red', 'blue', 'green', 'red', 'blue']).reshape(-1, 1)
ohe = OneHotEncoder(sparse_output=False)
encoded_ohe = ohe.fit_transform(colors)
print(f"\nOne-Hot Encoding:")
print(f"Categories: {ohe.categories_[0]}")
print(encoded_ohe)

---

<a id="pipelines"></a>

#### Scikit-learn Pipelines

Pipelines chain preprocessing + model into a single object, preventing data leakage and simplifying code.

| Benefit | Description |
| --- | --- |
| **No data leakage** | Scaler fit only on training data during cross-validation |
| **Clean code** | Single `.fit()` and `.predict()` call |
| **Reproducibility** | Entire workflow saved/loaded as one object |
| **ColumnTransformer** | Different preprocessing for numerical vs categorical features |

#### Cross-Validation Strategies

| Method | Description | When to Use |
| --- | --- | --- |
| **KFold** | Split into K equal folds, train on K-1, test on 1 | Default for most cases |
| **StratifiedKFold** | Preserves class distribution in each fold | Imbalanced classification |
| **LeaveOneOut** | Each sample is a test set once | Very small datasets |
| **TimeSeriesSplit** | Expanding window, respects temporal order | Time-series data |
| **RepeatedKFold** | Repeat KFold N times with different splits | More robust estimates |
| **GroupKFold** | Ensures same group not in train and test | Grouped data (patients, users) |

In [ ]:
# ──────────────────────────────────────────────
# Building a Pipeline
# ──────────────────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Create sample mixed-type dataset
from sklearn.datasets import load_iris
X, y = load_iris(return_X_y=True, as_frame=True)

# Simple pipeline: scale -> classify
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=200))
])

# Fit and predict
pipe.fit(X, y)
print("Pipeline steps:", [step[0] for step in pipe.steps])
print("Accuracy:", pipe.score(X, y).round(4))

In [ ]:
# ──────────────────────────────────────────────
# ColumnTransformer — Different preprocessing per column type
# ──────────────────────────────────────────────
from sklearn.compose import ColumnTransformer

# Simulate mixed-type data
df_mixed = pd.DataFrame({
    'age': [25, 30, np.nan, 40, 35],
    'salary': [50000, np.nan, 70000, 80000, 60000],
    'city': ['NYC', 'LA', 'NYC', 'SF', 'LA'],
    'education': ['BS', 'MS', 'PhD', 'BS', 'MS']
})
y_mixed = [0, 1, 1, 0, 1]

num_features = ['age', 'salary']
cat_features = ['city', 'education']

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]), num_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_features)
])

# Full pipeline with preprocessor + classifier
full_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=200))
])

full_pipe.fit(df_mixed, y_mixed)
print("ColumnTransformer pipeline fitted successfully!")
print(f"Feature names after transform: {preprocessor.get_feature_names_out().tolist()}")

In [ ]:
# ──────────────────────────────────────────────
# Cross-Validation
# ──────────────────────────────────────────────
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold

X, y = load_breast_cancer(return_X_y=True)

# Simple cross-validation
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000))
])

# 5-fold CV (stratified by default for classifiers)
scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
print(f"5-Fold CV Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")
print(f"Individual folds: {scores.round(4)}")

# Stratified K-Fold (explicit)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores_skf = cross_val_score(pipe, X, y, cv=skf, scoring='accuracy')
print(f"\n10-Fold Stratified CV: {scores_skf.mean():.4f} (+/- {scores_skf.std():.4f})")

# Multiple metrics
from sklearn.model_selection import cross_validate
results = cross_validate(pipe, X, y, cv=5,
                         scoring=['accuracy', 'precision', 'recall', 'f1'])
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    vals = results[f'test_{metric}']
    print(f"{metric:>10}: {vals.mean():.4f} (+/- {vals.std():.4f})")

In [ ]:
# ──────────────────────────────────────────────
# Bias-Variance Tradeoff Visualization
# ──────────────────────────────────────────────
import numpy as np

complexity = np.linspace(1, 10, 100)
bias = 1.0 / complexity
variance = 0.05 * complexity ** 1.5
total_error = bias + variance + 0.3  # irreducible error

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(complexity, bias, 'b-', linewidth=2, label='Bias^2')
ax.plot(complexity, variance, 'r-', linewidth=2, label='Variance')
ax.plot(complexity, total_error, 'k--', linewidth=2, label='Total Error')
ax.axhline(y=0.3, color='gray', linestyle=':', alpha=0.5, label='Irreducible Error')

# Mark optimum
opt_idx = np.argmin(total_error)
ax.axvline(x=complexity[opt_idx], color='green', linestyle='--', alpha=0.7)
ax.text(complexity[opt_idx]+0.2, max(total_error)*0.9, 'Optimal\nComplexity',
        fontsize=9, color='green', fontweight='bold')

ax.set_xlabel('Model Complexity')
ax.set_ylabel('Error')
ax.set_title('Bias-Variance Tradeoff', fontweight='bold')
ax.legend(loc='upper center')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ──────────────────────────────────────────────
# GridSearchCV and RandomizedSearchCV
# ──────────────────────────────────────────────
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import randint

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Grid Search — exhaustive (small param space)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10],
    'min_samples_split': [2, 5]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)
print("GridSearchCV:")
print(f"  Best params: {grid.best_params_}")
print(f"  Best CV score: {grid.best_score_:.4f}")
print(f"  Test score: {grid.score(X_test, y_test):.4f}")

# Randomized Search — faster (large param space)
param_dist = {
    'n_estimators': randint(50, 500),
    'max_depth': randint(3, 20),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10)
}

random_search = RandomizedSearchCV(RandomForestClassifier(random_state=42),
                                    param_dist, n_iter=20, cv=5,
                                    scoring='accuracy', n_jobs=-1, random_state=42)
random_search.fit(X_train, y_train)
print("\nRandomizedSearchCV:")
print(f"  Best params: {random_search.best_params_}")
print(f"  Best CV score: {random_search.best_score_:.4f}")
print(f"  Test score: {random_search.score(X_test, y_test):.4f}")

---

<a id="regression"></a>

### Part 2: Supervised Learning — Regression Models

#### Regression Algorithms Overview

| Algorithm | Type | Key Idea | Pros | Cons |
| --- | --- | --- | --- | --- |
| **Linear Regression** | Parametric | Fit a line minimizing MSE | Simple, interpretable, fast | Assumes linearity |
| **Ridge (L2)** | Regularized | Linear + L2 penalty | Handles multicollinearity | No feature selection |
| **Lasso (L1)** | Regularized | Linear + L1 penalty | Feature selection (sparse) | Unstable with correlated features |
| **ElasticNet** | Regularized | L1 + L2 combined | Best of both | Two hyperparams to tune |
| **Decision Tree** | Non-parametric | Recursive binary splits | Captures nonlinearity | Overfits easily |
| **Random Forest** | Ensemble | Bagging of decision trees | Robust, handles nonlinearity | Less interpretable |
| **SVR** | Kernel | Fit within epsilon tube | Effective in high dimensions | Slow on large datasets |
| **KNN Regressor** | Instance-based | Average of K nearest neighbors | No training needed | Slow prediction, curse of dimensionality |

In [ ]:
# ──────────────────────────────────────────────
# Regression Models — Complete Comparison
# ──────────────────────────────────────────────
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X, y = fetch_california_housing(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=1)': Ridge(alpha=1.0),
    'Lasso (alpha=0.1)': Lasso(alpha=0.1),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'KNN (k=5)': KNeighborsRegressor(n_neighbors=5),
}

print(f"{'Model':<25} {'RMSE':>8} {'MAE':>8} {'R2':>8}")
print("-" * 55)

for name, model in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"{name:<25} {rmse:>8.4f} {mae:>8.4f} {r2:>8.4f}")

In [ ]:
# ──────────────────────────────────────────────
# Regularization Comparison (Ridge vs Lasso vs ElasticNet)
# ──────────────────────────────────────────────
import numpy as np

X, y = fetch_california_housing(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

feature_names = fetch_california_housing().feature_names

print("Coefficient comparison (regularization effect):\n")
print(f"{'Feature':<15} {'Linear':>10} {'Ridge':>10} {'Lasso':>10}")
print("-" * 50)

lr = LinearRegression().fit(X_train_s, y_train)
ridge = Ridge(alpha=1.0).fit(X_train_s, y_train)
lasso = Lasso(alpha=0.1).fit(X_train_s, y_train)

for i, name in enumerate(feature_names):
    print(f"{name:<15} {lr.coef_[i]:>10.4f} {ridge.coef_[i]:>10.4f} {lasso.coef_[i]:>10.4f}")

print(f"\n{'Non-zero coefs:':<15} {np.sum(lr.coef_ != 0):>10} {np.sum(ridge.coef_ != 0):>10} {np.sum(lasso.coef_ != 0):>10}")
print("\nNotice: Lasso drives some coefficients toward zero (feature selection)")

---

<a id="classification"></a>

### Part 3: Supervised Learning — Classification Models

#### Classification Algorithms Overview

| Algorithm | Type | Key Idea | Pros | Cons |
| --- | --- | --- | --- | --- |
| **Logistic Regression** | Linear | Sigmoid function for probability | Fast, interpretable, good baseline | Assumes linear decision boundary |
| **Decision Tree** | Non-parametric | Recursive splits on features | Interpretable, handles mixed data | Overfits, unstable |
| **Random Forest** | Ensemble (Bagging) | Many trees, majority vote | Robust, handles nonlinearity | Slow, less interpretable |
| **Gradient Boosting** | Ensemble (Boosting) | Sequential error correction | Very accurate | Slow training, prone to overfit |
| **SVM** | Kernel-based | Maximum margin hyperplane | Effective in high dims | Slow on large data, needs scaling |
| **KNN** | Instance-based | Vote of K nearest neighbors | Simple, no training | Slow prediction, needs scaling |
| **Naive Bayes** | Probabilistic | Bayes theorem + independence | Fast, good for text, small data | Strong independence assumption |
| **AdaBoost** | Ensemble (Boosting) | Focus on misclassified samples | Less overfitting than GB | Sensitive to noise/outliers |

In [ ]:
# ──────────────────────────────────────────────
# Classification Models — Complete Comparison
# ──────────────────────────────────────────────
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              AdaBoostClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
}

print(f"{'Model':<25} {'Accuracy':>10} {'F1':>10} {'AUC':>10}")
print("-" * 60)

for name, model in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    print(f"{name:<25} {acc:>10.4f} {f1:>10.4f} {auc:>10.4f}")

In [ ]:
# ──────────────────────────────────────────────
# Confusion Matrix + Classification Report
# ──────────────────────────────────────────────
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = Pipeline([('scaler', StandardScaler()),
                 ('clf', RandomForestClassifier(n_estimators=100, random_state=42))])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(f"  TN={cm[0,0]}  FP={cm[0,1]}")
print(f"  FN={cm[1,0]}  TP={cm[1,1]}")

# Plot
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
                                         display_labels=['Malignant', 'Benign'], ax=ax,
                                         cmap='Blues')
ax.set_title('Confusion Matrix — Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
# ──────────────────────────────────────────────
# ROC Curve Comparison
# ──────────────────────────────────────────────
from sklearn.metrics import roc_curve, auc

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

fig, ax = plt.subplots(figsize=(7, 5))

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True),
    'KNN': KNeighborsClassifier(),
}

for name, clf in classifiers.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', clf)])
    pipe.fit(X_train, y_train)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ──────────────────────────────────────────────
# Decision Boundary Visualization (2D)
# ──────────────────────────────────────────────
from sklearn.decomposition import PCA

X, y = load_breast_cancer(return_X_y=True)

# Reduce to 2D for visualization
pca = PCA(n_components=2)
X_2d = pca.fit_transform(StandardScaler().fit_transform(X))
X_train, X_test, y_train, y_test = train_test_split(X_2d, y, test_size=0.2, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

models = {
    'Logistic Regression': LogisticRegression(),
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf'),
}

for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_train, y_train)
    
    # Create mesh grid
    xx, yy = np.meshgrid(
        np.linspace(X_2d[:, 0].min()-1, X_2d[:, 0].max()+1, 200),
        np.linspace(X_2d[:, 1].min()-1, X_2d[:, 1].max()+1, 200)
    )
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='RdYlBu', edgecolors='k', s=20)
    ax.set_title(f'{name}\nAcc={model.score(X_test, y_test):.3f}')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

plt.tight_layout()
plt.show()

---

<a id="ensemble"></a>

### Part 4: Ensemble Methods — Bagging and Boosting

#### Ensemble Methods Overview

| Method | Strategy | Variance | Bias | Examples |
| --- | --- | --- | --- | --- |
| **Bagging** | Parallel, bootstrap aggregation | Reduces variance | No effect | Random Forest, BaggingClassifier |
| **Boosting** | Sequential, error correction | May increase | Reduces bias | XGBoost, LightGBM, AdaBoost, CatBoost |
| **Stacking** | Meta-learner on base predictions | Reduces both | Reduces both | StackingClassifier |
| **Voting** | Combine predictions (avg/vote) | Reduces variance | No effect | VotingClassifier |

In [ ]:
# ──────────────────────────────────────────────
# Ensemble Methods Comparison
# ──────────────────────────────────────────────
from sklearn.ensemble import (VotingClassifier, BaggingClassifier,
                              StackingClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier, RandomForestClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Bagging
bagging = Pipeline([('scaler', StandardScaler()),
                    ('clf', BaggingClassifier(estimator=DecisionTreeClassifier(),
                                             n_estimators=50, random_state=42))])

# Voting (soft)
voting = Pipeline([('scaler', StandardScaler()),
                   ('clf', VotingClassifier(estimators=[
                       ('lr', LogisticRegression(max_iter=1000)),
                       ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
                       ('svm', SVC(probability=True))
                   ], voting='soft'))])

# Stacking
stacking = Pipeline([('scaler', StandardScaler()),
                     ('clf', StackingClassifier(estimators=[
                         ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
                         ('svm', SVC(probability=True))
                     ], final_estimator=LogisticRegression()))])

ensembles = {
    'Bagging (DT)': bagging,
    'Voting (Soft)': voting,
    'Stacking (RF+SVM+LR)': stacking,
}

print(f"{'Ensemble':<25} {'Accuracy':>10}")
print("-" * 40)
for name, ens in ensembles.items():
    ens.fit(X_train, y_train)
    acc = ens.score(X_test, y_test)
    print(f"{name:<25} {acc:>10.4f}")

---

<a id="unsupervised"></a>

### Part 5: Unsupervised Learning — Clustering and Dimensionality Reduction

#### Clustering Algorithms

| Algorithm | Type | Key Idea | Pros | Cons |
| --- | --- | --- | --- | --- |
| **KMeans** | Centroid-based | Assign points to nearest centroid, update centroids | Fast, scalable | Must specify K, assumes spherical clusters |
| **DBSCAN** | Density-based | Group dense regions, outliers are noise | No K needed, finds arbitrary shapes | Sensitive to eps and min_samples |
| **Hierarchical** | Agglomerative | Bottom-up merging of closest clusters | Dendrogram visualization, no K needed | Slow O(n^3), not for large data |
| **Mean Shift** | Density-based | Shift points toward density peaks | No K needed, finds modes | Slow, bandwidth is critical |

#### Dimensionality Reduction

| Method | Type | Key Idea | Use Case |
| --- | --- | --- | --- |
| **PCA** | Linear | Project onto directions of max variance | Feature reduction, visualization, preprocessing |
| **t-SNE** | Non-linear | Preserve local distances in low dims | 2D/3D visualization only (not for preprocessing) |
| **UMAP** | Non-linear | Preserve both local and global structure | Visualization, faster than t-SNE |
| **LDA** | Supervised linear | Maximize class separability | Classification preprocessing |

In [ ]:
# ──────────────────────────────────────────────
# KMeans Clustering with Elbow Method
# ──────────────────────────────────────────────
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

# Generate clustered data
X_blobs, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

# Elbow method to find optimal K
inertias = []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_blobs)
    inertias.append(km.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Elbow plot
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (Within-cluster SS)')
axes[0].set_title('Elbow Method')
axes[0].axvline(x=4, color='r', linestyle='--', label='Optimal K=4')
axes[0].legend()

# Clustering result
km = KMeans(n_clusters=4, n_init=10, random_state=42)
labels = km.fit_predict(X_blobs)
axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap='viridis', s=20)
axes[1].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
               c='red', marker='X', s=200, label='Centroids')
axes[1].set_title(f'KMeans (K=4)')
axes[1].legend()

plt.tight_layout()
plt.show()

# Silhouette score
from sklearn.metrics import silhouette_score
sil = silhouette_score(X_blobs, labels)
print(f"Silhouette Score: {sil:.4f} (closer to 1 = better)")

In [ ]:
# ──────────────────────────────────────────────
# DBSCAN — Density-Based Clustering
# ──────────────────────────────────────────────
from sklearn.cluster import DBSCAN

# Create data with noise and non-spherical shapes
from sklearn.datasets import make_moons
X_moons, y_moons = make_moons(n_samples=300, noise=0.1, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# KMeans on moons (fails)
km = KMeans(n_clusters=2, n_init=10, random_state=42)
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=km.fit_predict(X_moons), cmap='viridis', s=20)
axes[0].set_title('KMeans (fails on non-spherical)')

# DBSCAN on moons (works)
db = DBSCAN(eps=0.2, min_samples=5)
labels_db = db.fit_predict(X_moons)
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_db, cmap='viridis', s=20)
axes[1].set_title(f'DBSCAN (eps=0.2, found {len(set(labels_db))-1} clusters)')

# Hierarchical
from sklearn.cluster import AgglomerativeClustering
hc = AgglomerativeClustering(n_clusters=2, linkage='average')
axes[2].scatter(X_moons[:, 0], X_moons[:, 1], c=hc.fit_predict(X_moons), cmap='viridis', s=20)
axes[2].set_title('Agglomerative Clustering')

plt.suptitle('Clustering Comparison on Non-Spherical Data', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ──────────────────────────────────────────────
# PCA — Dimensionality Reduction
# ──────────────────────────────────────────────
from sklearn.decomposition import PCA

X, y = load_breast_cancer(return_X_y=True)
X_scaled = StandardScaler().fit_transform(X)
print(f"Original shape: {X.shape} ({X.shape[1]} features)")

# Full PCA to see explained variance
pca_full = PCA().fit(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Explained variance
axes[0].bar(range(1, 11), pca_full.explained_variance_ratio_[:10])
axes[0].plot(range(1, 11), np.cumsum(pca_full.explained_variance_ratio_[:10]), 'ro-')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('PCA Explained Variance')
axes[0].axhline(y=0.95, color='g', linestyle='--', label='95% threshold')
axes[0].legend()

# 2D visualization
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled)
scatter = axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='RdYlBu', s=20, alpha=0.7)
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})')
axes[1].set_title('PCA 2D Projection — Breast Cancer')
plt.colorbar(scatter, ax=axes[1], label='Target')

plt.tight_layout()
plt.show()

n_95 = np.argmax(np.cumsum(pca_full.explained_variance_ratio_) >= 0.95) + 1
print(f"Components for 95% variance: {n_95} (from {X.shape[1]} features)")

In [ ]:
# ──────────────────────────────────────────────
# t-SNE vs PCA Visualization
# ──────────────────────────────────────────────
from sklearn.manifold import TSNE

X, y = load_digits(return_X_y=True)
X_scaled = StandardScaler().fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# PCA
pca_2d = PCA(n_components=2).fit_transform(X_scaled)
axes[0].scatter(pca_2d[:, 0], pca_2d[:, 1], c=y, cmap='tab10', s=5, alpha=0.6)
axes[0].set_title('PCA — Digits Dataset')

# t-SNE
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(X_scaled)
axes[1].scatter(tsne_2d[:, 0], tsne_2d[:, 1], c=y, cmap='tab10', s=5, alpha=0.6)
axes[1].set_title('t-SNE — Digits Dataset')

plt.suptitle('PCA vs t-SNE for High-Dimensional Data', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("t-SNE preserves local structure better than PCA for visualization")

---

<a id="xgboost"></a>

### Part 6: XGBoost — Extreme Gradient Boosting

#### XGBoost Key Features

| Feature | Description |
| --- | --- |
| **Regularization** | L1 (alpha) and L2 (lambda) built-in to prevent overfitting |
| **Tree pruning** | Max depth-based pruning (depth-first); more efficient |
| **Missing values** | Learns best direction for missing values automatically |
| **Parallel processing** | Parallelized tree construction at feature level |
| **Cache-aware** | Optimized data structure for faster training |
| **Out-of-core** | Can train on data larger than memory |

#### Key Hyperparameters

| Parameter | Description | Typical Range |
| --- | --- | --- |
| **n_estimators** | Number of boosting rounds | 100-1000 |
| **learning_rate (eta)** | Step size shrinkage | 0.01-0.3 |
| **max_depth** | Maximum tree depth | 3-10 |
| **min_child_weight** | Minimum sum of instance weight in child | 1-10 |
| **subsample** | Fraction of samples per tree | 0.6-1.0 |
| **colsample_bytree** | Fraction of features per tree | 0.6-1.0 |
| **gamma** | Minimum loss reduction for split | 0-5 |
| **reg_alpha** | L1 regularization | 0-1 |
| **reg_lambda** | L2 regularization | 1-10 |

In [ ]:
# ──────────────────────────────────────────────
# XGBoost — Classification
# ──────────────────────────────────────────────
try:
    import xgboost as xgb
    print(f"XGBoost version: {xgb.__version__}")
    
    X, y = load_breast_cancer(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # XGBoost Classifier
    xgb_clf = xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        min_child_weight=1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    )
    
    xgb_clf.fit(X_train, y_train,
                eval_set=[(X_test, y_test)],
                verbose=False)
    
    y_pred = xgb_clf.predict(X_test)
    y_prob = xgb_clf.predict_proba(X_test)[:, 1]
    
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
    print(f"AUC-ROC:   {roc_auc_score(y_test, y_prob):.4f}")
    
    # Feature importance
    fi = pd.Series(xgb_clf.feature_importances_,
                   index=load_breast_cancer().feature_names).nlargest(10)
    print(f"\nTop 10 features:")
    for feat, imp in fi.items():
        print(f"  {feat:<25} {imp:.4f}")
        
except ImportError:
    print("XGBoost not installed. Run: pip install xgboost")

---

<a id="lightgbm"></a>

#### LightGBM — Light Gradient Boosting Machine

| Feature | LightGBM vs XGBoost |
| --- | --- |
| **Tree growth** | Leaf-wise (best-first) vs XGBoost level-wise | 
| **Speed** | Faster due to histogram-based splitting |
| **Memory** | Lower memory usage (histogram binning) |
| **Categorical** | Native categorical feature support |
| **Large data** | Better for very large datasets |
| **GOSS** | Gradient-based One-Side Sampling for speed |
| **EFB** | Exclusive Feature Bundling for sparse features |

In [ ]:
# ──────────────────────────────────────────────
# LightGBM — Classification
# ──────────────────────────────────────────────
try:
    import lightgbm as lgb
    print(f"LightGBM version: {lgb.__version__}")
    
    X, y = load_breast_cancer(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    lgb_clf = lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        num_leaves=31,       # key param for leaf-wise growth
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        verbose=-1
    )
    
    lgb_clf.fit(X_train, y_train,
                eval_set=[(X_test, y_test)],
                callbacks=[lgb.log_evaluation(0)])
    
    y_pred = lgb_clf.predict(X_test)
    y_prob = lgb_clf.predict_proba(X_test)[:, 1]
    
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
    print(f"AUC-ROC:   {roc_auc_score(y_test, y_prob):.4f}")
    
except ImportError:
    print("LightGBM not installed. Run: pip install lightgbm")

---

<a id="catboost"></a>

#### CatBoost — Categorical Boosting

| Feature | Description |
| --- | --- |
| **Categorical handling** | Native support — no encoding needed |
| **Ordered boosting** | Reduces prediction shift (overfitting technique unique to CatBoost) |
| **Symmetric trees** | Balanced trees for faster inference |
| **GPU training** | Built-in GPU support |
| **Default performance** | Best out-of-the-box performance (needs less tuning) |

In [ ]:
# ──────────────────────────────────────────────
# CatBoost — Classification
# ──────────────────────────────────────────────
try:
    from catboost import CatBoostClassifier
    
    X, y = load_breast_cancer(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    cat_clf = CatBoostClassifier(
        iterations=200,
        learning_rate=0.1,
        depth=5,
        l2_leaf_reg=3.0,
        random_seed=42,
        verbose=0
    )
    
    cat_clf.fit(X_train, y_train, eval_set=(X_test, y_test))
    
    y_pred = cat_clf.predict(X_test)
    y_prob = cat_clf.predict_proba(X_test)[:, 1]
    
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
    print(f"AUC-ROC:   {roc_auc_score(y_test, y_prob):.4f}")
    
except ImportError:
    print("CatBoost not installed. Run: pip install catboost")

---

<a id="boosting-comparison"></a>

#### Boosting Methods Comparison

| Feature | XGBoost | LightGBM | CatBoost |
| --- | --- | --- | --- |
| **Tree growth** | Level-wise | Leaf-wise | Symmetric |
| **Speed** | Fast | Fastest | Moderate |
| **Categorical** | Encoding needed | Basic support | Native (best) |
| **Default perf** | Good | Good | Best (less tuning) |
| **Memory** | Higher | Lowest | Moderate |
| **GPU** | Yes | Yes | Yes (best) |
| **Missing values** | Auto-handled | Auto-handled | Auto-handled |
| **When to use** | General purpose | Large data, speed | Categorical features, quick baseline |

---

<a id="optuna"></a>

### Part 7: Optuna — Hyperparameter Optimization

#### Optuna Overview

| Feature | Description |
| --- | --- |
| **Define-by-Run** | Dynamically construct search spaces with if/else logic |
| **Sampler** | TPE (Tree-structured Parzen Estimator) — Bayesian optimization |
| **Pruning** | Early termination of unpromising trials (MedianPruner, HyperbandPruner) |
| **Multi-objective** | Optimize multiple metrics simultaneously |
| **Dashboard** | Built-in visualization with optuna.visualization |
| **Storage** | SQLite/MySQL for distributed optimization and resuming studies |

#### Optuna vs GridSearch vs RandomSearch

| Feature | GridSearch | RandomSearch | Optuna (TPE) |
| --- | --- | --- | --- |
| **Strategy** | Exhaustive | Random sampling | Bayesian (informed) |
| **Efficiency** | O(n^k) | User-defined budget | Learns from past trials |
| **Dynamic spaces** | No | No | Yes (conditional params) |
| **Pruning** | No | No | Yes (early stopping) |
| **Scalability** | Poor | Good | Best |

In [ ]:
# ──────────────────────────────────────────────
# Optuna — Hyperparameter Optimization for XGBoost
# ──────────────────────────────────────────────
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    
    X, y = load_breast_cancer(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        }
        
        from sklearn.ensemble import GradientBoostingClassifier
        model = GradientBoostingClassifier(**{k: v for k, v in params.items() 
                                             if k in ['n_estimators', 'learning_rate', 'max_depth', 'subsample']},
                                           random_state=42)
        
        scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
        return scores.mean()
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=30, show_progress_bar=False)
    
    print(f"Best trial accuracy: {study.best_value:.4f}")
    print(f"Best parameters:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")
    
    print(f"\nTotal trials: {len(study.trials)}")
    
except ImportError:
    print("Optuna not installed. Run: pip install optuna")

In [ ]:
# ──────────────────────────────────────────────
# Optuna — Pruning (Early Stopping of Bad Trials)
# ──────────────────────────────────────────────
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    
    X, y = load_breast_cancer(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    def objective_with_pruning(trial):
        n_estimators = trial.suggest_int('n_estimators', 50, 500)
        lr = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 10)
        
        model = GradientBoostingClassifier(
            n_estimators=n_estimators, learning_rate=lr,
            max_depth=max_depth, random_state=42
        )
        
        # Step-wise evaluation for pruning
        from sklearn.model_selection import StratifiedKFold
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        
        scores = []
        for step, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
            model.fit(X_train[train_idx], y_train[train_idx])
            score = model.score(X_train[val_idx], y_train[val_idx])
            scores.append(score)
            
            # Report intermediate value for pruning
            trial.report(np.mean(scores), step)
            
            # Prune if this trial is unpromising
            if trial.should_prune():
                raise optuna.TrialPruned()
        
        return np.mean(scores)
    
    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
    study = optuna.create_study(direction='maximize', pruner=pruner)
    study.optimize(objective_with_pruning, n_trials=20, show_progress_bar=False)
    
    pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
    complete = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
    
    print(f"Completed trials: {complete}")
    print(f"Pruned trials: {pruned}")
    print(f"Best accuracy: {study.best_value:.4f}")
    
except ImportError:
    print("Optuna not installed. Run: pip install optuna")

---

<a id="shap"></a>

### Part 8: SHAP — Model Explainability

#### SHAP (SHapley Additive exPlanations) Overview

| Concept | Description |
| --- | --- |
| **SHAP values** | Contribution of each feature to a prediction, based on game theory (Shapley values) |
| **Additivity** | SHAP values sum to the difference between prediction and expected value |
| **Consistency** | If a feature's contribution increases, its SHAP value never decreases |
| **Local** | Explains individual predictions (why did THIS prediction happen?) |
| **Global** | Aggregate SHAP values to see overall feature importance |

#### SHAP Explainer Types

| Explainer | Use With | Speed |
| --- | --- | --- |
| **TreeExplainer** | Tree-based models (XGBoost, RF, LightGBM) | Fast |
| **KernelExplainer** | Any model (model-agnostic) | Slow |
| **LinearExplainer** | Linear models | Fast |
| **DeepExplainer** | Deep learning models (PyTorch/TF) | Moderate |

In [ ]:
# ──────────────────────────────────────────────
# SHAP — Feature Importance and Explanation
# ──────────────────────────────────────────────
try:
    import shap
    
    X, y = load_breast_cancer(return_X_y=True, as_frame=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train a Random Forest
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # SHAP TreeExplainer
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    
    # Summary plot (global importance)
    print("SHAP Summary Plot — Global Feature Importance")
    shap.summary_plot(shap_values[1], X_test, plot_type='bar', show=True, max_display=15)
    
except ImportError:
    print("SHAP not installed. Run: pip install shap")

In [ ]:
# ──────────────────────────────────────────────
# SHAP — Individual Prediction Explanation
# ──────────────────────────────────────────────
try:
    import shap
    
    # Explain a single prediction
    idx = 0
    print(f"Prediction for sample {idx}: {model.predict(X_test.iloc[[idx]])[0]}")
    print(f"Actual label: {y_test.iloc[idx]}")
    print(f"Probability: {model.predict_proba(X_test.iloc[[idx]])[0]}")
    
    # Waterfall plot for individual explanation
    shap.initjs()
    # Force plot for single prediction
    shap.force_plot(explainer.expected_value[1], 
                    shap_values[1][idx], 
                    X_test.iloc[idx],
                    matplotlib=True, show=True)
    
    # Beeswarm plot (detailed global view)
    print("\nBeeswarm plot — feature impact distribution:")
    shap.summary_plot(shap_values[1], X_test, show=True, max_display=15)
    
except Exception as e:
    print(f"SHAP visualization: {e}")

---

<a id="mlflow"></a>

### Part 9: MLflow — Experiment Tracking and Model Registry

#### MLflow Components

| Component | Purpose |
| --- | --- |
| **Tracking** | Log parameters, metrics, artifacts for each experiment run |
| **Models** | Package models in standard format for deployment |
| **Registry** | Centralized model store with versioning (Staging, Production, Archived) |
| **Projects** | Package code for reproducible runs |

#### MLflow Tracking Concepts

| Concept | Description |
| --- | --- |
| **Experiment** | Named group of runs (e.g., "breast_cancer_classification") |
| **Run** | Single execution with logged params, metrics, artifacts |
| **Parameter** | Input config (learning_rate, max_depth) |
| **Metric** | Output measure (accuracy, loss) — can be logged over steps |
| **Artifact** | Output file (model, plot, data sample) |
| **Tag** | Metadata label (team, version, notes) |

In [ ]:
# ──────────────────────────────────────────────
# MLflow — Experiment Tracking
# ──────────────────────────────────────────────
try:
    import mlflow
    import mlflow.sklearn
    
    # Set local tracking (no server needed)
    mlflow.set_tracking_uri("file:///tmp/mlflow_demo")
    mlflow.set_experiment("breast_cancer_classification")
    
    X, y = load_breast_cancer(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    models_to_compare = {
        'LogisticRegression': LogisticRegression(max_iter=1000),
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
        'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    }
    
    for name, model in models_to_compare.items():
        with mlflow.start_run(run_name=name):
            # Train
            pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
            pipe.fit(X_train, y_train)
            y_pred = pipe.predict(X_test)
            
            # Log parameters
            mlflow.log_param("model_type", name)
            mlflow.log_param("scaler", "StandardScaler")
            
            # Log metrics
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)
            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_score", f1)
            
            # Log model
            mlflow.sklearn.log_model(pipe, "model")
            
            print(f"{name:<25} Accuracy={acc:.4f}  F1={f1:.4f}")
    
    print("\nMLflow experiment logged to /tmp/mlflow_demo")
    print("Run 'mlflow ui --backend-store-uri file:///tmp/mlflow_demo' to view")
    
except ImportError:
    print("MLflow not installed. Run: pip install mlflow")

---

<a id="wandb"></a>

### Part 10: Weights and Biases — Experiment Management

#### W&B Overview

| Feature | Description |
| --- | --- |
| **Experiment Tracking** | Log metrics, hyperparameters, system metrics automatically |
| **Sweeps** | Automated hyperparameter optimization (Bayesian, Random, Grid) |
| **Artifacts** | Version datasets, models, and other files |
| **Tables** | Interactive data tables for visualization |
| **Reports** | Collaborative dashboards and documentation |
| **Alerts** | Automated notifications on metric conditions |

In [ ]:
# ──────────────────────────────────────────────
# Weights & Biases — Experiment Tracking (Demo)
# ──────────────────────────────────────────────
try:
    import wandb
    
    # Initialize in offline mode (no API key needed for demo)
    wandb.init(project="ml_core_demo", mode="offline", config={
        "model": "RandomForest",
        "n_estimators": 100,
        "max_depth": 10,
        "dataset": "breast_cancer"
    })
    
    X, y = load_breast_cancer(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Log metrics
    wandb.log({"accuracy": acc, "f1_score": f1})
    
    # Log confusion matrix
    wandb.log({"confusion_matrix": wandb.plot.confusion_matrix(
        y_true=y_test, preds=y_pred,
        class_names=["Malignant", "Benign"]
    )})
    
    wandb.finish()
    print(f"W&B logged: Accuracy={acc:.4f}, F1={f1:.4f}")
    print("Run 'wandb sync' to upload offline runs")
    
except ImportError:
    print("W&B not installed. Run: pip install wandb")
except Exception as e:
    print(f"W&B demo: {e}")

---

<a id="ml-fundamentals-qa"></a>

#### Machine Learning Fundamentals — Interview Questions (Part 1)

| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 1 | What is ML and how does it differ from traditional programming? | ML, definition | • Traditional: rules + data = output<br>• ML: data + output = rules (learned from data)<br>• ML systems improve with experience without explicit programming |
| 2 | Relationship between AI, ML, and DL? | AI, ML, DL | • AI: broadest (any intelligent behavior)<br>• ML: subset of AI (learns from data)<br>• DL: subset of ML (neural networks with many layers) |
| 3 | Main types of ML? | Supervised, unsupervised | • Supervised: labeled data (classification, regression)<br>• Unsupervised: no labels (clustering, dimensionality reduction)<br>• Semi-supervised: few labels + much unlabeled<br>• Reinforcement: learns via rewards/penalties |
| 4 | Supervised learning examples? | Supervised, examples | • Spam detection (email -> spam/not spam)<br>• House price prediction (features -> price)<br>• Medical diagnosis (symptoms -> disease) |
| 5 | Unsupervised learning examples? | Unsupervised, examples | • Customer segmentation (group similar customers)<br>• Anomaly detection (find unusual patterns)<br>• Topic modeling (discover themes in documents) |
| 6 | What is semi-supervised learning? | Semi-supervised | • Few labeled + many unlabeled samples<br>• Model learns structure from unlabeled, refines with labeled<br>• Useful when labeling is expensive (medical, legal) |
| 7 | What is reinforcement learning? | RL, reward | • Agent learns by interacting with environment<br>• Gets rewards for good actions, penalties for bad<br>• Use cases: game playing, robotics, recommendation |
| 8 | What is self-supervised learning? | Self-supervised | • Creates labels from the data itself (no manual labeling)<br>• BERT: predict masked words; GPT: predict next token<br>• Foundation of modern LLMs and vision models |
| 9 | Classification vs regression? | Classification, regression | • Classification: discrete categories (spam/not spam)<br>• Regression: continuous values (price, temperature)<br>• Look at target variable type to decide |
| 10 | What is overfitting? | Overfitting, detection | • Model memorizes training data, fails on new data<br>• Detect: high train acc, low test acc; large gap<br>• Prevent: regularization, dropout, more data, early stopping, cross-validation |
| 11 | What is underfitting? | Underfitting | • Model too simple to capture patterns<br>• Both train and test performance poor<br>• Fix: more features, complex model, less regularization, longer training |
| 12 | Bias-variance tradeoff? | Bias, variance | • Bias: error from wrong assumptions (underfitting)<br>• Variance: sensitivity to training data (overfitting)<br>• Total error = bias^2 + variance + irreducible noise<br>• Goal: find sweet spot (optimal model complexity) |
| 13 | High bias vs high variance? | Bias, variance | • High bias: consistently wrong predictions (underfit)<br>• High variance: predictions change a lot with different data (overfit)<br>• Simple models = high bias; Complex models = high variance |
| 14 | Curse of dimensionality? | Dimensionality, curse | • As features increase, data becomes sparse in high-D space<br>• Distance metrics become meaningless<br>• Need exponentially more data; solution: feature selection, PCA |
| 15 | Why cross-validation? | CV, validation | • Single train-test split may not be representative<br>• CV gives more robust performance estimate<br>• K-fold: split data into K parts, each serves as test once |
| 16 | K-fold CV — how and typical K? | K-fold | • Split data into K folds; train on K-1, test on 1; repeat K times<br>• Average results across folds<br>• K=5 or K=10 most common; K=5 balances bias-variance of estimate |
| 17 | Stratified CV? | Stratified, imbalanced | • Maintains class proportions in each fold<br>• Critical for imbalanced datasets<br>• Use over standard K-fold when classes are unequal |
| 18 | Train-test-validation split ratio? | Split, ratio | • Common: 70/15/15 or 80/10/10<br>• Large datasets: 98/1/1 is fine (millions of samples)<br>• Validation for hyperparameter tuning; test for final evaluation |
| 19 | What is data leakage? | Leakage | • Training model on information it shouldn't have<br>• Examples: scaling before split, using future data, target leakage<br>• Silently inflates metrics; model fails in production |
| 20 | Parameters vs hyperparameters? | Parameters, hyperparameters | • Parameters: learned during training (weights, biases)<br>• Hyperparameters: set before training (lr, epochs, max_depth)<br>• Hyperparameters control the learning process |
| 21 | Hyperparameter tuning approaches? | Tuning, search | • Grid search: exhaustive (exponential time)<br>• Random search: samples randomly (often better than grid)<br>• Bayesian optimization: models objective (Optuna, most efficient) |
| 22 | Grid vs Random vs Bayesian? | Grid, random, Bayesian | • Grid: try all combos; good for small search spaces<br>• Random: samples uniformly; faster, often finds good configs<br>• Bayesian: learns from past trials; most sample-efficient |
| 23 | What is ensemble learning? | Ensemble | • Combine multiple models for better predictions<br>• Reduces variance (bagging) or bias (boosting)<br>• Usually outperforms any single model |
| 24 | How does bagging reduce variance? | Bagging, variance | • Train models on bootstrap samples (random subsets with replacement)<br>• Average predictions; reduces variance<br>• Random Forest = bagging + feature randomness |
| 25 | How does boosting reduce bias? | Boosting, bias | • Sequential models, each focuses on errors of previous<br>• Converts weak learners to strong learner<br>• AdaBoost, XGBoost, LightGBM, CatBoost |

---

#### Machine Learning Fundamentals — Interview Questions (Part 2)

| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 26 | What is stacking? | Stacking, meta | • Train diverse base models (Level 0)<br>• Meta-learner combines their predictions (Level 1)<br>• More complex but can be very powerful |
| 27 | Bagging vs boosting? | Bagging, boosting | • Bagging: parallel, reduces variance (Random Forest)<br>• Boosting: sequential, reduces bias (XGBoost)<br>• Boosting more prone to overfitting if not regularized |
| 28 | Feature engineering importance? | Feature eng | • Often more impactful than model choice<br>• Domain knowledge creates informative features<br>• Good features can make simple models outperform complex ones |
| 29 | Feature selection methods? | Selection, filter | • Filter: statistical tests (correlation, chi-squared) — fast<br>• Wrapper: use model performance (forward/backward selection)<br>• Embedded: built into model (L1 regularization, tree importance) |
| 30 | When dimensionality reduction? | Dim reduction, PCA | • Too many features relative to samples<br>• Multicollinearity issues<br>• Visualization (reduce to 2-3D)<br>• Speed up training |
| 31 | What is regularization? | Regularization | • Adds penalty to loss function to constrain model complexity<br>• Prevents overfitting; encourages simpler models<br>• L1 (Lasso), L2 (Ridge), ElasticNet, Dropout |
| 32 | L1 vs L2 regularization? | L1, L2 | • L1 (Lasso): pushes weights to exactly 0 (feature selection)<br>• L2 (Ridge): shrinks weights toward 0 (never exactly 0)<br>• L1 = diamond constraint; L2 = circular constraint<br>• L1 produces sparse models |
| 33 | What is ElasticNet? | ElasticNet | • Combines L1 + L2 regularization<br>• Use when many correlated features (L1 alone picks randomly)<br>• alpha controls L1/L2 ratio |
| 34 | Good train, bad test — what to do? | Overfitting, fix | • Classic overfitting; model memorized training data<br>• Actions: add regularization, reduce model complexity, get more data<br>• Use cross-validation, early stopping, dropout |
| 35 | No Free Lunch theorem? | NFL | • No algorithm is universally best for all problems<br>• Must match algorithm to problem structure<br>• Try multiple approaches; domain knowledge matters |
| 36 | Occam's Razor in model selection? | Occam, simplicity | • Prefer simpler models that perform equally well<br>• Simpler = more interpretable, less prone to overfit<br>• Complexity only justified by significant improvement |
| 37 | Generative vs discriminative models? | Generative, discriminative | • Generative: model P(X,Y) — learn data distribution (Naive Bayes, GMM, VAE)<br>• Discriminative: model P(Y&#124;X) directly — learn decision boundary (Logistic Reg, SVM, neural nets) |
| 38 | What is MLE? | MLE, likelihood | • Maximum Likelihood Estimation: find parameters that maximize P(data&#124;params)<br>• Choose params that make observed data most probable<br>• Foundation of logistic regression, many ML models |
| 39 | MLE vs MAP? | MLE, MAP | • MLE: maximize P(data&#124;params) — no prior<br>• MAP: maximize P(params&#124;data) = P(data&#124;params)*P(params) — includes prior<br>• MAP with Gaussian prior = L2 regularization |
| 40 | Very limited labeled data — what to do? | Few labels, limited | • Semi-supervised learning (use unlabeled data)<br>• Transfer learning (pretrained models)<br>• Data augmentation; active learning (label most informative samples)<br>• Self-supervised pretraining, few-shot learning |
| 41 | Parametric vs non-parametric? | Parametric | • Parametric: fixed number of params (Linear Reg, Logistic Reg)<br>• Non-parametric: params grow with data (KNN, Decision Tree, KDE)<br>• Parametric = assumptions; Non-parametric = flexible |
| 42 | Online vs batch learning? | Online, batch | • Batch: train on entire dataset at once<br>• Online: update with each new sample (streaming data)<br>• Online useful when data arrives continuously or too large for memory |
| 43 | What is active learning? | Active learning | • Model selects most informative samples for labeling<br>• Reduces labeling cost by querying uncertain examples<br>• Useful when labeling is expensive |
| 44 | How to detect concept drift? | Concept drift | • Monitor model performance over time<br>• Statistical tests on feature distributions (PSI, KS test)<br>• Retrain trigger: performance drops below threshold |
| 45 | Interpretability vs accuracy? | Interpretability | • Simple models (linear, tree) = interpretable but less accurate<br>• Complex (ensemble, NN) = accurate but black-box<br>• Use SHAP/LIME for interpretability of complex models<br>• Regulated domains may require interpretable models |
| 46 | What is a learning curve? | Learning curve | • Plot train/test error vs training set size<br>• Diagnoses: high bias (both errors high), high variance (big gap)<br>• More data helps high variance; more features help high bias |
| 47 | How to choose a loss function? | Loss function | • Regression: MSE (smooth), MAE (robust to outliers), Huber (compromise)<br>• Classification: Cross-entropy (standard), Focal (imbalanced)<br>• Match loss to problem characteristics |
| 48 | What is multi-task learning? | Multi-task | • Single model learns multiple related tasks simultaneously<br>• Shared representations improve generalization<br>• Example: joint NER + POS tagging; age + gender prediction |
| 49 | Is model production-ready? | Production | • Meets performance thresholds on diverse test sets<br>• Handles edge cases and adversarial inputs<br>• Latency/throughput within requirements; monitoring in place<br>• Reproducible, versioned, documented |
| 50 | What is the VC dimension? | VC dimension | • Maximum number of points a model can perfectly classify (shatter)<br>• Measures model capacity/complexity<br>• Linear classifier in 2D has VC=3 |

---

<a id="preprocessing-qa"></a>

#### Data Preprocessing and Feature Engineering — Interview Questions

| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 1 | Why is preprocessing important? | Preprocessing | • Garbage in, garbage out — raw data is noisy, incomplete<br>• Models assume clean, numeric input<br>• Skipping preprocessing leads to poor or misleading results |
| 2 | Strategies for missing data? | Missing, imputation | • Drop: if small % missing and random (MCAR)<br>• Impute: mean/median/mode, KNN imputation, iterative<br>• Flag: add binary indicator column for missingness<br>• Model-based: use algorithms that handle missing (XGBoost) |
| 3 | Mean vs median vs KNN imputation? | Imputation, mean | • Mean: simple, sensitive to outliers<br>• Median: robust to outliers; better for skewed data<br>• KNN: uses similar samples; better but slower |
| 4 | Detect and handle outliers? | Outliers, detection | • IQR method: below Q1-1.5*IQR or above Q3+1.5*IQR<br>• Z-score: more than 3 standard deviations<br>• Handle: remove, cap/floor (winsorize), or transform (log) |
| 5 | IQR vs z-score for outliers? | IQR, z-score | • IQR: non-parametric, robust, works on any distribution<br>• Z-score: assumes normal distribution<br>• IQR better for skewed data; z-score for roughly normal |
| 6 | When to keep outliers? | Outliers, keep | • Domain-specific valid data (fraud amounts, earthquake magnitudes)<br>• When outliers represent important signal, not noise<br>• Removing may bias the model |
| 7 | Why feature scaling? | Scaling | • Gradient-based algorithms need scaled features (neural nets, SVM, KNN)<br>• Distance-based methods affected by magnitude<br>• Tree-based models don't need scaling |
| 8 | Normalization vs standardization? | Normalization, standardization | • Normalization (Min-Max): scale to [0,1]; use when bounded range<br>• Standardization (Z-score): mean=0, std=1; use for most cases<br>• StandardScaler most common in practice |
| 9 | Min-Max vs Standard vs Robust scaler? | Scaler, comparison | • Min-Max: bounded [0,1]; sensitive to outliers<br>• Standard: centered, unit variance; mildly affected by outliers<br>• Robust: uses median and IQR; best when outliers present |
| 10 | Encoding categorical variables? | Encoding, categorical | • One-hot: binary columns per category (few categories)<br>• Label: ordinal numbers (for tree-based or ordinal data)<br>• Target: mean of target per category (watch for leakage) |
| 11 | One-hot vs label vs target encoding? | One-hot, label | • One-hot: no ordinal assumption; bad for high-cardinality<br>• Label: compact but implies order (only for trees/ordinal)<br>• Target: powerful but leaks info without proper regularization |
| 12 | Common ordinal encoding mistake? | Ordinal, mistake | • Using ordinal encoding for non-ordinal categories<br>• Model interprets numbers as ordered/continuous<br>• Only use when natural order exists (low/med/high) |
| 13 | Target encoding leakage? | Target encoding, leakage | • Directly uses target value, causing info leak<br>• Fix: use cross-validation encoding (fold-based)<br>• Or use smoothing to regularize toward global mean |
| 14 | High-cardinality categoricals? | High-cardinality | • Too many categories for one-hot (1000+ cities)<br>• Solutions: target encoding, frequency encoding, embeddings<br>• Group rare categories into "Other" |
| 15 | Strategies for class imbalance? | Imbalance, SMOTE | • Resample: oversample minority (SMOTE) or undersample majority<br>• Class weights: weight loss function by inverse frequency<br>• Use appropriate metrics: F1, AUPRC, not accuracy<br>• Algorithm-level: Focal Loss |
| 16 | SMOTE and when it fails? | SMOTE, synthetic | • Creates synthetic minority samples between nearest neighbors<br>• Fails: when minority class is not well-separated, noisy data<br>• Fails with high-dimensional sparse data<br>• Variants: ADASYN, Borderline-SMOTE address some issues |
| 17 | Oversampling vs undersampling? | Oversampling, undersampling | • Oversampling: keeps all data, may overfit minority patterns<br>• Undersampling: loses majority data, may lose information<br>• Best: combine both, or use class weights in loss function |
| 18 | PCA — how and assumptions? | PCA, assumptions | • Finds directions of maximum variance (principal components)<br>• Linear projection; assumes linear relationships<br>• Features should be scaled; components are orthogonal |
| 19 | How many PCA components? | PCA, components | • Plot cumulative explained variance; choose threshold (95% typical)<br>• Elbow in scree plot<br>• Cross-validate downstream task performance |
| 20 | t-SNE vs UMAP? | t-SNE, UMAP | • Both for visualization (2-3D)<br>• UMAP: faster, preserves global structure better, scalable<br>• t-SNE: better for local structure; slower; non-deterministic<br>• Neither for preprocessing; only visualization |
| 21 | Handle multicollinearity? | Multicollinearity, VIF | • Calculate VIF (Variance Inflation Factor); VIF > 5-10 = problem<br>• Remove one of correlated pair; PCA<br>• Regularization (Ridge) handles it automatically |
| 22 | Feature interaction? | Interaction, polynomial | • Combine features: age * income, ratio features<br>• Polynomial features for non-linear relationships<br>• Domain knowledge drives best interactions |
| 23 | Time-series feature differences? | Time series | • Lag features (value at t-1, t-2, etc.)<br>• Rolling statistics (moving avg, std)<br>• Date decomposition (day of week, month, holiday)<br>• Must respect temporal order — never shuffle |
| 24 | Feature extraction vs selection? | Extraction, selection | • Extraction: create new representations (PCA, autoencoders)<br>• Selection: pick subset of existing features (filter, wrapper)<br>• Extraction changes features; selection keeps originals |
| 25 | Preprocessing before split — why? | Split, leakage | • Fitting scaler/encoder on full data leaks test info<br>• Always fit on training only, then transform both train and test<br>• Use Pipeline for correct ordering |
| 26 | Text data as features? | Text, features | • TF-IDF or BoW for traditional ML<br>• Pretrained embeddings (BERT) for deep learning<br>• n-grams, sentiment scores, text length as engineered features |
| 27 | What is binning? | Binning, discretization | • Convert continuous to categorical (age -> age_group)<br>• Useful when relationship is monotonic but non-linear<br>• Risk: lose granularity and information |
| 28 | Validate preprocessing pipeline? | Validate, pipeline | • Compare feature distributions train vs test<br>• Check for unexpected NaN/inf after transform<br>• Use Pipeline to ensure correct ordering and prevent leakage |
| 29 | Data augmentation techniques? | Augmentation | • Images: flip, rotate, crop, color jitter, Cutout, MixUp<br>• Text: synonym replacement, back-translation, paraphrase<br>• Tabular: SMOTE, noise injection |
| 30 | Frequency and binary encoding? | Frequency, binary | • Frequency: replace category with its count/proportion in data<br>• Binary: encode category ID in binary digits (log2 columns)<br>• Both handle high-cardinality better than one-hot |

---

<a id="metrics-qa"></a>

#### Evaluation Metrics — Interview Questions

| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 1 | When is accuracy misleading? | Accuracy, misleading | • Imbalanced datasets: 99% accuracy by predicting majority class<br>• Example: 1% fraud — predicting "no fraud" always = 99% accuracy<br>• Use precision, recall, F1, AUC instead |
| 2 | Precision vs recall tradeoff? | Precision, recall | • Precision: of predicted positives, how many correct? TP/(TP+FP)<br>• Recall: of actual positives, how many found? TP/(TP+FN)<br>• High precision = few false alarms; High recall = find all positives |
| 3 | F1 vs F-beta? | F1, F-beta | • F1: harmonic mean of precision and recall (equal weight)<br>• F-beta: beta > 1 weights recall more; beta < 1 weights precision more<br>• F2 for medical (miss fewer cases); F0.5 for spam (fewer false positives) |
| 4 | Confusion matrix walkthrough? | Confusion matrix | • TP: correctly predicted positive; FP: wrongly predicted positive<br>• FN: missed positive; TN: correctly predicted negative<br>• Rows = actual; Columns = predicted (or vice versa depending on library) |
| 5 | What is specificity? | Specificity, TNR | • True Negative Rate: TN/(TN+FP)<br>• How well model identifies negatives (healthy patients correctly cleared)<br>• Important in medical screening to avoid unnecessary treatment |
| 6 | How to interpret ROC curve? | ROC, curve | • Plot TPR (recall) vs FPR at different thresholds<br>• Top-left corner = perfect classifier<br>• Diagonal = random guessing<br>• Area under curve (AUC) summarizes performance |
| 7 | What is AUC-ROC? | AUC, ROC | • Probability that model ranks random positive higher than random negative<br>• AUC=0.5: random; AUC=1.0: perfect<br>• Scale-invariant; threshold-independent |
| 8 | PR curve vs ROC curve? | PR curve, imbalanced | • PR curve: precision vs recall (better for imbalanced data)<br>• ROC can look optimistic with class imbalance (FPR stays low)<br>• Use PR curve when positive class is rare |
| 9 | What is average precision? | AP, average precision | • Area under PR curve (weighted mean of precisions at each threshold)<br>• Summarizes PR curve in single number<br>• Better than F1 when you want threshold-free evaluation |
| 10 | MAE, MSE, RMSE — when each? | MAE, MSE, RMSE | • MAE: robust to outliers, interpretable (same units)<br>• MSE: penalizes large errors more (differentiable)<br>• RMSE: MSE in original units; most commonly reported |
| 11 | R-squared and negative R^2? | R-squared | • Proportion of variance explained by model (0 to 1)<br>• Negative R^2: model worse than predicting mean<br>• Does not mean model accuracy is negative |
| 12 | Adjusted R-squared? | Adjusted R2 | • Penalizes adding features that don't improve fit<br>• Increases only if new feature improves model<br>• Preferred over R^2 for comparing models with different feature counts |
| 13 | What is log loss? | Log loss, probabilistic | • -mean(y*log(p) + (1-y)*log(1-p)) for binary<br>• Penalizes confident wrong predictions heavily<br>• Evaluates probability quality, not just predictions |
| 14 | Cohen's Kappa? | Kappa, agreement | • Measures agreement adjusted for chance<br>• Better than accuracy for imbalanced classes<br>• Kappa=0: chance agreement; Kappa=1: perfect |
| 15 | Matthews Correlation Coefficient? | MCC | • Balanced metric that works even with imbalanced data<br>• Range: -1 to +1; 0 = random; considers all four confusion matrix cells<br>• Often best single-number metric for binary classification |
| 16 | What is mAP in object detection? | mAP, detection | • Mean Average Precision across all classes<br>• AP computed at different IoU thresholds (0.5, 0.75)<br>• mAP@0.5 vs mAP@[0.5:0.95] for different strictness |
| 17 | What is IoU? | IoU, intersection | • Intersection over Union of predicted vs ground truth boxes<br>• IoU > 0.5 typically considered a correct detection<br>• Used in object detection and segmentation |
| 18 | Dice coefficient? | Dice, segmentation | • 2*overlap / (pred_size + gt_size); equivalent to F1 for sets<br>• Commonly used in medical image segmentation<br>• Range 0 to 1; sensitive to small objects |
| 19 | What is perplexity? | Perplexity, LM | • 2^(cross-entropy); measures how well LM predicts text<br>• Lower = better predictions<br>• PPL=1: perfect; PPL=vocab_size: random |
| 20 | BLEU score and limitations? | BLEU | • Compares n-gram overlap between generated and reference text<br>• Limitations: ignores meaning, favors short outputs, sensitive to single reference<br>• Used for machine translation evaluation |
| 21 | ROUGE vs BLEU? | ROUGE, BLEU | • ROUGE: recall-oriented (how much reference is captured)<br>• BLEU: precision-oriented (how much generation is correct)<br>• ROUGE for summarization; BLEU for translation |
| 22 | Choose right metric for business? | Metric, business | • Define cost of errors: false positive vs false negative<br>• Fraud: optimize recall (catch all fraud)<br>• Spam: optimize precision (don't block good email)<br>• Regression: MAE if outliers matter less, MSE if large errors costly |
| 23 | ML model calibration? | Calibration | • Predicted probability should match actual frequency<br>• If model says 80% confident, should be right 80% of time<br>• Check: reliability diagram; fix: Platt scaling, isotonic regression |
| 24 | What is a lift chart? | Lift, marketing | • Shows model improvement over random selection<br>• Top 10% of model's predictions contain how many positives?<br>• Lift = model precision / baseline rate |
| 25 | KS statistic? | KS, kolmogorov | • Max separation between cumulative distributions of positives and negatives<br>• Higher KS = better class separation<br>• Used in credit scoring, marketing |
| 26 | Different costs for FP and FN? | Cost, asymmetric | • Use cost-sensitive learning: weight classes in loss function<br>• Adjust classification threshold<br>• Optimize business metric directly, not just accuracy |
| 27 | Cross-entropy vs log loss? | Cross-entropy, log loss | • Binary log loss = binary cross-entropy (same formula)<br>• Cross-entropy generalizes to multi-class<br>• Both measure quality of probabilistic predictions |
| 28 | Evaluating clustering without labels? | Clustering, evaluation | • Silhouette score: cohesion vs separation (-1 to 1)<br>• Davies-Bouldin index: lower is better<br>• Calinski-Harabasz: ratio of between to within cluster variance |
| 29 | What is silhouette score? | Silhouette | • (b-a)/max(a,b) where a=intra-cluster dist, b=nearest-cluster dist<br>• Range [-1, 1]; >0.5 good; <0 likely wrong cluster<br>• Average across all samples |
| 30 | Top-k accuracy? | Top-k | • Correct if true label in top-k predictions<br>• Used in ImageNet (top-5 accuracy standard)<br>• Relevant for multi-class with many similar classes |

---

<a id="stats-qa"></a>

#### Probability and Statistics — Interview Questions

| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 1 | Bayes' theorem in ML? | Bayes | • P(A&#124;B) = P(B&#124;A)*P(A)/P(B)<br>• Foundation of Naive Bayes, Bayesian optimization<br>• Update beliefs (prior to posterior) given evidence |
| 2 | Frequentist vs Bayesian? | Frequentist, Bayesian | • Frequentist: probability = long-run frequency; fixed params<br>• Bayesian: probability = degree of belief; params have distributions<br>• Bayesian includes prior knowledge; updates with data |
| 3 | Gaussian vs Bernoulli vs Poisson? | Distribution | • Gaussian: continuous, bell-curve (heights, errors)<br>• Bernoulli: binary outcome (coin flip, click/no-click)<br>• Poisson: count of events in interval (web visits/hour) |
| 4 | Central Limit Theorem? | CLT | • Sample means approach normal distribution as n increases<br>• Works regardless of original distribution (n>30 rule of thumb)<br>• Foundation for confidence intervals and hypothesis testing |
| 5 | What is a p-value? | P-value | • Probability of observing data as extreme under null hypothesis<br>• p < 0.05: reject null (but arbitrary threshold)<br>• Does NOT measure probability hypothesis is true |
| 6 | Type I vs Type II errors? | Type I, Type II | • Type I (alpha): false positive — reject true null hypothesis<br>• Type II (beta): false negative — fail to reject false null<br>• Power = 1-beta = probability of detecting real effect |
| 7 | Covariance vs correlation? | Covariance, correlation | • Covariance: direction of linear relationship (unbounded units)<br>• Correlation: normalized covariance [-1, 1] (unit-free)<br>• Correlation does NOT imply causation |
| 8 | Conditional probability and Naive Bayes? | Conditional | • P(A&#124;B): probability of A given B occurred<br>• Naive Bayes assumes features are conditionally independent given class<br>• "Naive" = independence assumption (often violated but still works) |
| 9 | Entropy in decision trees? | Entropy, info theory | • Measures impurity: -sum(p_i * log2(p_i))<br>• Max entropy = uniform distribution (most uncertain)<br>• Information gain = parent entropy minus weighted child entropy |
| 10 | Cross-entropy vs KL divergence? | KL divergence | • Cross-entropy: H(p,q) = -sum(p*log(q)) — loss function<br>• KL divergence: D(p&#124;&#124;q) = H(p,q) - H(p)<br>• KL measures how q differs from p; cross-entropy = KL + constant |
| 11 | Mutual information? | Mutual info | • How much knowing one variable reduces uncertainty about another<br>• MI(X,Y) = H(X) + H(Y) - H(X,Y)<br>• Used for feature selection; captures non-linear dependencies |
| 12 | Likelihood vs probability? | Likelihood | • Probability: P(data&#124;fixed params) — function of data<br>• Likelihood: L(params&#124;fixed data) — function of parameters<br>• Same formula but different perspective |
| 13 | Prior, posterior, likelihood? | Prior, posterior | • Prior: P(params) — belief before data<br>• Likelihood: P(data&#124;params) — data given params<br>• Posterior: P(params&#124;data) — updated belief after data |
| 14 | Confidence vs credible interval? | Confidence, credible | • Confidence (frequentist): 95% of such intervals contain true param<br>• Credible (Bayesian): 95% probability param is in interval<br>• Bayesian interpretation more intuitive |
| 15 | What is bootstrapping? | Bootstrap | • Sample with replacement from dataset; compute statistic<br>• Repeat many times to get distribution of statistic<br>• Estimates uncertainty without assumptions about distribution |
| 16 | Markov property? | Markov | • Future depends only on current state, not history<br>• Foundation: HMMs, Markov chains, MDPs in RL<br>• Simplifying assumption for sequential modeling |
| 17 | Monte Carlo sampling? | Monte Carlo | • Random sampling to estimate intractable quantities<br>• MCMC: sample from complex posterior distributions<br>• Used in Bayesian inference, policy evaluation in RL |
| 18 | EM algorithm? | EM, expectation | • E-step: estimate hidden variables given params<br>• M-step: maximize params given hidden variable estimates<br>• Used in GMMs, missing data problems |
| 19 | Test for normality? | Normality, Shapiro | • Shapiro-Wilk test, Q-Q plot, D'Agostino test<br>• Matters for: t-tests, linear regression assumptions<br>• Many ML methods don't require normality (trees, boosting) |
| 20 | Chi-squared test? | Chi-squared | • Tests independence between two categorical variables<br>• Compares observed vs expected frequencies<br>• Used for feature selection with categorical target |
| 21 | Simpson's paradox? | Simpson | • Trend reverses when data is grouped vs ungrouped<br>• Confounding variable causes misleading conclusion<br>• Must consider lurking variables in analysis |
| 22 | Hypothesis testing for models? | Hypothesis, test | • Paired t-test comparing two models' CV scores<br>• McNemar's test for comparing classifiers<br>• Corrected resampled t-test accounts for CV non-independence |
| 23 | Law of large numbers? | LLN | • Sample average converges to expected value as n increases<br>• More data = more reliable estimates<br>• Foundation for why larger training sets improve models |
| 24 | Multivariate Gaussian? | Multivariate, Gaussian | • Generalizes normal distribution to higher dimensions<br>• Characterized by mean vector and covariance matrix<br>• Used in: GMMs, LDA, anomaly detection (Mahalanobis distance) |
| 25 | Sufficient statistics? | Sufficient | • Statistics that capture all information about a parameter<br>• Sample mean + variance are sufficient for Gaussian<br>• Useful for data compression and efficient estimation |

---

<a id="algorithms-qa"></a>

#### Classical ML Algorithms — Interview Questions

| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 1 | Linear Regression assumptions? | Linear, assumptions | • Linear relationship between features and target<br>• Homoscedasticity (constant variance of errors)<br>• Independence of errors; no multicollinearity<br>• Normally distributed residuals (for inference) |
| 2 | Polynomial Regression? | Polynomial | • Add polynomial terms (x^2, x^3) to capture non-linearity<br>• Still "linear" in parameters (linear regression with polynomial features)<br>• Risk of overfitting with high degree |
| 3 | Ridge vs Lasso — when Lasso does feature selection? | Ridge, Lasso | • Lasso (L1): pushes coefficients to exactly zero<br>• Ridge (L2): shrinks but never zeros out<br>• Lasso selects features when few features are truly important |
| 4 | Why is Logistic Regression called regression? | Logistic | • Predicts log-odds (continuous), then converts to probability via sigmoid<br>• "Regression" refers to the linear model underneath<br>• Output is probability; threshold converts to classification |
| 5 | Sigmoid in Logistic Regression? | Sigmoid, logistic | • Squashes linear output to (0,1) range<br>• sigma(z) = 1/(1+e^-z)<br>• Smooth, differentiable; enables gradient-based optimization |
| 6 | Decision Trees — how to split? | Split, entropy | • Try each feature at each threshold; pick best split<br>• Criterion: maximize info gain (entropy) or Gini reduction<br>• Greedy: locally optimal at each node |
| 7 | Entropy vs Gini impurity? | Entropy, Gini | • Both measure node impurity; very similar in practice<br>• Gini: faster to compute (no log); sklearn default<br>• Entropy: slightly more balanced splits |
| 8 | What is pruning? | Pruning, tree | • Remove branches that add complexity without improving validation<br>• Pre-pruning: max_depth, min_samples_split<br>• Post-pruning: grow full tree then remove branches (cost-complexity) |
| 9 | Random Forest vs single tree? | Random Forest | • RF = bagging + feature subsampling at each split<br>• Much lower variance; more robust to overfitting<br>• Trade-off: loses interpretability of single tree |
| 10 | Feature importance in RF? | Importance, RF | • Mean decrease in impurity (Gini) per feature across trees<br>• Permutation importance: drop in score when feature shuffled<br>• Permutation more reliable but slower |
| 11 | XGBoost — what makes it different? | XGBoost | • Regularization built-in (L1+L2 on leaves)<br>• Second-order gradients (Newton method)<br>• Sparsity-aware splits; column subsampling<br>• System optimizations: cache-aware, parallel |
| 12 | XGBoost vs LightGBM vs CatBoost? | XGBoost, LightGBM | • XGBoost: level-wise growth; robust, widely used<br>• LightGBM: leaf-wise growth; faster on large data<br>• CatBoost: native categorical handling; ordered boosting<br>• LightGBM fastest; CatBoost best for categoricals |
| 13 | LightGBM speed advantage? | LightGBM, speed | • Leaf-wise growth (not level-wise) — deeper trees faster<br>• Gradient-based one-side sampling (GOSS)<br>• Exclusive feature bundling (EFB); histogram binning |
| 14 | CatBoost categorical features? | CatBoost, categorical | • Ordered target statistics: avoids leakage via permutation<br>• No need for preprocessing categoricals<br>• Built-in handling without one-hot encoding |
| 15 | Gradient Boosting step by step? | GBM, boosting | • 1. Start with initial prediction (mean)<br>• 2. Compute residuals (pseudo-residuals)<br>• 3. Fit new tree to residuals<br>• 4. Add tree prediction scaled by learning rate<br>• 5. Repeat |
| 16 | AdaBoost vs Gradient Boosting? | AdaBoost, GBM | • AdaBoost: reweight misclassified samples<br>• GBM: fit to gradient of loss (more general)<br>• GBM more powerful; AdaBoost simpler and original |
| 17 | SVM maximum margin? | SVM, margin | • Find hyperplane that maximizes distance to nearest points (support vectors)<br>• Maximum margin = most robust decision boundary<br>• Only support vectors determine the boundary |
| 18 | Kernel trick in SVM? | Kernel, trick | • Maps data to higher dimension without computing the transformation<br>• Computes dot products in high-D implicitly<br>• Makes non-linearly separable data separable |
| 19 | Linear vs polynomial vs RBF kernels? | Kernel, RBF | • Linear: linearly separable data (fastest)<br>• Polynomial: captures feature interactions (degree d)<br>• RBF: most flexible, infinite-dimensional mapping; default choice |
| 20 | KNN — how and weaknesses? | KNN | • Classifies by majority vote of K nearest neighbors<br>• Weaknesses: slow at inference (must search all points), curse of dimensionality<br>• Sensitive to feature scaling and irrelevant features |
| 21 | Choose K in KNN? | KNN, K value | • Small K: noisy (overfit); Large K: smooth (underfit)<br>• Try odd K for binary classification (avoid ties)<br>• Cross-validation to select optimal K |
| 22 | Naive Bayes assumption? | Naive Byes, assumption | • Features are conditionally independent given class<br>• Almost never true in practice, but works surprisingly well<br>• Fails when features are highly correlated |
| 23 | Gaussian vs Multinomial vs Bernoulli NB? | NB, variants | • Gaussian: continuous features (assume normal distribution)<br>• Multinomial: discrete counts (text classification, word frequencies)<br>• Bernoulli: binary features (word presence/absence) |
| 24 | K-Means limitations? | K-Means, limitations | • Assumes spherical clusters of equal size<br>• Sensitive to initialization (use K-means++)<br>• Must specify K in advance<br>• Can't handle non-convex shapes |
| 25 | Choose K in K-Means? | K-Means, elbow | • Elbow method: plot inertia vs K, pick elbow<br>• Silhouette analysis: higher is better<br>• Domain knowledge; BIC/AIC for GMMs |
| 26 | Elbow method failure? | Elbow, fail | • No clear elbow when clusters are not well-separated<br>• Subjective interpretation<br>• Use silhouette score as alternative validation |
| 27 | DBSCAN handling shapes? | DBSCAN, density | • Density-based: groups nearby points, finds arbitrary shapes<br>• Params: eps (radius), min_samples (minimum points)<br>• Automatically detects cluster count; identifies noise/outliers |
| 28 | K-Means vs DBSCAN vs hierarchical? | Clustering, comparison | • K-Means: fast, spherical clusters, need K<br>• DBSCAN: arbitrary shapes, finds outliers, need eps<br>• Hierarchical: dendrogram, no K needed, slow on large data |
| 29 | Gaussian Mixture Models vs K-Means? | GMM, K-Means | • GMM: soft assignment (probabilities), allows elliptical clusters<br>• K-Means: hard assignment, spherical only<br>• GMM is probabilistic generalization of K-Means |
| 30 | EM in GMM? | EM, GMM | • E-step: compute probability each point belongs to each Gaussian<br>• M-step: update means, covariances, mixing weights<br>• Iterate until convergence |
| 31 | Simple vs complex model? | Simple, complex | • Start simple (logistic regression) as baseline<br>• Add complexity if baseline insufficient<br>• Simple = interpretable, faster, less overfit risk<br>• Complex justified by significant accuracy gain |
| 32 | Algorithms for millions of rows? | Scale, large data | • LightGBM: histogram-based, very fast<br>• SGD classifier/regressor: online learning, mini-batches<br>• Avoid: KNN (O(n) prediction), kernel SVM (O(n^2-n^3) training) |
| 33 | Hierarchical clustering? | Hierarchical, dendrogram | • Agglomerative: bottom-up (merge closest pairs)<br>• Divisive: top-down (split largest cluster)<br>• Dendrogram visualizes merge history; cut at desired level |
| 34 | Information gain? | Info gain, split | • Reduction in entropy after splitting<br>• IG = H(parent) - weighted_sum(H(children))<br>• Higher IG = better split; used by ID3, C4.5 |
| 35 | Multinomial Logistic Regression? | Multinomial, softmax | • Softmax function extends sigmoid to multi-class<br>• One set of weights per class<br>• Cross-entropy loss for multi-class |

---

<a id="master-cheatsheet"></a>

### Master Interview Cheat Sheet — Machine Learning

| # | Question | Answer |
|---|----------|--------|
| 1 | What is ML? | • Learning patterns from data without explicit programming; AI subset |
| 2 | Supervised vs unsupervised? | • Supervised: labeled data; unsupervised: no labels |
| 3 | Overfitting? | • Memorizes training; fix: regularization, dropout, more data |
| 4 | Bias-variance tradeoff? | • Total error = bias^2 + variance + noise; sweet spot needed |
| 5 | Cross-validation? | • K-fold: train K times, each fold as test; K=5 typical |
| 6 | Data leakage? | • Training on info model shouldn't have; silently inflates metrics |
| 7 | Feature scaling? | • StandardScaler most common; needed for gradient-based and distance-based |
| 8 | Missing data? | • Mean/median impute, KNN, or model-based; flag missingness |
| 9 | Class imbalance? | • SMOTE, class weights, appropriate metrics (F1, AUC-PR) |
| 10 | L1 vs L2? | • L1: sparse (feature selection); L2: shrinks (no zeros) |
| 11 | Accuracy misleading? | • Imbalanced data; predict majority class = high accuracy |
| 12 | Precision vs recall? | • Precision: few false alarms; recall: find all positives |
| 13 | F1 score? | • Harmonic mean of precision and recall |
| 14 | AUC-ROC? | • Probability random positive ranked higher than negative |
| 15 | MAE vs MSE? | • MAE: robust to outliers; MSE: penalizes large errors more |
| 16 | Log loss? | • Measures quality of probabilistic predictions |
| 17 | Bayes' theorem? | • P(A&#124;B) = P(B&#124;A)*P(A)/P(B); update beliefs with evidence |
| 18 | p-value? | • Probability of data under null; not probability of hypothesis |
| 19 | Type I vs II errors? | • I: false positive; II: false negative |
| 20 | Correlation != causation | • Correlation measures linear association, not causal link |
| 21 | Linear Regression assumptions? | • Linearity, homoscedasticity, independence, normal residuals |
| 22 | Logistic Regression? | • Linear model + sigmoid for classification; predicts probability |
| 23 | Decision Tree splitting? | • Maximize info gain or Gini reduction; greedy at each node |
| 24 | Random Forest? | • Bagging + feature subsampling; reduces variance |
| 25 | XGBoost? | • Regularized GBM, second-order gradients; top performer |
| 26 | LightGBM? | • Leaf-wise growth; GOSS + EFB; fastest for large data |
| 27 | CatBoost? | • Native categoricals; ordered boosting; less tuning needed |
| 28 | SVM? | • Maximum margin hyperplane; kernel trick for non-linear |
| 29 | KNN? | • Majority vote of K nearest; lazy learner; slow at inference |
| 30 | Naive Bayes? | • P(class&#124;features); assumes feature independence; fast, works well |
| 31 | K-Means? | • Assign to nearest centroid; iterate; need K, spherical clusters |
| 32 | DBSCAN? | • Density-based; arbitrary shapes; auto K; identifies noise |
| 33 | PCA? | • Project to max variance directions; linear; scale features first |
| 34 | t-SNE vs UMAP? | • Both for visualization; UMAP faster, preserves global structure |
| 35 | Ensemble methods? | • Combine models; bagging (variance), boosting (bias), stacking (meta) |
| 36 | Grid vs Random vs Bayesian tuning? | • Grid: exhaustive; Random: faster; Bayesian: most efficient |
| 37 | Regularization? | • Penalty on complexity; prevents overfitting |
| 38 | Feature selection? | • Filter (stats), wrapper (model-based), embedded (L1, tree importance) |
| 39 | One-hot vs Label encoding? | • One-hot: no order implied; Label: compact but implies order |
| 40 | Train-test split first? | • Always split BEFORE preprocessing to prevent leakage |
| 41 | Elbow method? | • Plot inertia vs K; pick bend; subjective; use silhouette also |
| 42 | Silhouette score? | • (b-a)/max(a,b); measures cluster quality; range [-1,1] |
| 43 | MLE? | • Find params maximizing P(data&#124;params) |
| 44 | Entropy? | • Measures impurity/uncertainty; max at uniform distribution |
| 45 | Bootstrapping? | • Sample with replacement; estimate statistic uncertainty |
| 46 | Feature engineering? | • Create informative features; often more important than model choice |
| 47 | Multicollinearity? | • Correlated features; VIF > 5-10; fix: remove, PCA, Ridge |
| 48 | Learning curve? | • Train/test error vs data size; diagnose bias vs variance |
| 49 | No Free Lunch? | • No universal best algorithm; match to problem |
| 50 | MCC? | • Best single metric for binary; considers all confusion matrix cells |